# Multi-Objective Bayesian Optimization for Fast-Charging Protocols

This notebook implements multi-objective Bayesian optimization to find Pareto-optimal fast-charging protocols that balance competing objectives:

---

## Objectives:

1. **Maximize charge throughput** (Q30) - Amount of charge stored in 30 minutes [Ah]
2. **Minimize total LLI** (total_lli) - Total lithium inventory loss from plating + SEI [mol]
3. **Minimize SEI growth** (sei_growth) - SEI thickness increase [nm]

---

## Configuration:

- **Cell**: LG M50 (NMC-graphite, OKane2022 parameters)
- **Degradation modes**: Reversible lithium plating + SEI growth
- **SOC range**: 10-90% (realistic user scenario)
- **Protocol structure**: 3-step with 10-minute intervals
- **Optimization budget**: 150 evaluations (50 iterations × 3 trials/iteration)
- **Method**: qNEHVI (quasi-Expected Hypervolume Improvement)

---

## 1. Setup and Imports

In [ ]:
# Add parent directory to path
import sys
sys.path.insert(0, '..')

# Core imports
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import warnings
import threading

# Ax Platform for multi-objective optimization
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.service.utils.report_utils import exp_to_df

# Our modules
from utils.pybamm_simulator import PyBaMMSimulator

# Suppress PyBaMM convergence warnings (they're usually harmless)
warnings.filterwarnings('ignore', message='.*corrector convergence.*')
warnings.filterwarnings('ignore', message='.*linesearch algorithm.*')

print("✓ Imports successful")
print(f"✓ Working directory: {sys.path[0]}")


## 2. Initialize PyBaMM Simulator

In [ ]:
# Initialize simulator with plating + SEI
simulator = PyBaMMSimulator(
    degradation_modes=['plating', 'SEI'],
    soc_start=0.1,  # 10% SOC
    soc_end=0.9,    # 90% SOC
    charge_time_minutes=30.0
)

print("Simulator Configuration:")
print(f"  Degradation modes: {simulator.degradation_modes}")
print(f"  SOC range: {simulator.soc_start*100:.0f}% - {simulator.soc_end*100:.0f}%")
print(f"  Time window: {simulator.charge_time_minutes} min")
print(f"  Parameter set: {simulator.parameter_set}")

## 3. Define Evaluation Function

This function takes protocol parameters from Ax and returns all three objectives:

In [ ]:
_PENALTY = {'charge_throughput': 0.0, 'total_lli': 1.0, 'sei_growth': 1000.0}
_EVAL_TIMEOUT = 120  # seconds


def evaluate_protocol(parameters):
    """
    Evaluate a charging protocol and return multi-objective metrics.

    Args:
        parameters: Dictionary with keys 'C1', 'C2', 'C3' (C-rates for each step)

    Returns:
        Dictionary with:
            - 'charge_throughput': Q30 [Ah] (to maximize)
            - 'total_lli': Total lithium loss [mol] (to minimize)
            - 'sei_growth': SEI thickness increase [nm] (to minimize)
    """
    c_rates = [parameters['C1'], parameters['C2'], parameters['C3']]
    step_durations = [10.0, 10.0, 10.0]

    result_container = [None]

    def _run():
        result_container[0] = simulator.run_and_extract(
            c_rates=c_rates,
            step_durations=step_durations,
            verbose=False,
        )

    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    thread.join(timeout=_EVAL_TIMEOUT)

    if thread.is_alive():
        print(f"  WARNING: simulation timed out after {_EVAL_TIMEOUT}s for {parameters}")
        return _PENALTY

    metrics = result_container[0]
    if not metrics['success']:
        return _PENALTY

    return {
        'charge_throughput': float(metrics['Q30']),
        'total_lli': float(metrics['total_lli']),
        'sei_growth': float(metrics['sei_growth']),
    }


# Test evaluation function
print("Testing evaluation function with 1C-1C-1C protocol...")
test_result = evaluate_protocol({'C1': 1.0, 'C2': 1.0, 'C3': 1.0})
print(f"✓ Test result: {test_result}")


## 4. Configure Multi-Objective Optimization

Set up Ax client with:
- Parameter space: C1, C2, C3 ∈ [0, 3]C
- Three objectives with different optimization directions
- Reference point for Pareto dominance

In [ ]:
# Initialize Ax client for multi-objective optimization
ax_client = AxClient()

# Create experiment
ax_client.create_experiment(
    name="fast_charge_multi_obj",
    parameters=[
        {"name": "C1", "type": "range", "bounds": [0.0, 3.0], "value_type": "float"},
        {"name": "C2", "type": "range", "bounds": [0.0, 3.0], "value_type": "float"},
        {"name": "C3", "type": "range", "bounds": [0.0, 3.0], "value_type": "float"},
    ],
    objectives={
        # Thresholds define the reference point for Pareto dominance
        "charge_throughput": ObjectiveProperties(minimize=False, threshold=0.02),
        "total_lli": ObjectiveProperties(minimize=True, threshold=0.025),
        "sei_growth": ObjectiveProperties(minimize=True, threshold=70.0),
    },
)

print("✓ Multi-objective experiment configured")
print(f"  Parameters: C1, C2, C3 ∈ [0, 3]C")
print(f"  Objectives:")
print(f"    - charge_throughput (maximize, threshold >= 0.02 Ah)")
print(f"    - total_lli (minimize, threshold <= 0.025 mol)")
print(f"    - sei_growth (minimize, threshold <= 70 nm)")

## 5. Add Baseline Trials (Warm Start)

Initialize the Gaussian Process with known good protocols from baseline evaluation:

In [ ]:
# Load baseline results if available
try:
    baseline_df = pd.read_csv('results/baseline_results.csv')
    print(f"✓ Loaded {len(baseline_df)} baseline results for warm start")
    
    # Add relevant baselines as initial trials
    # Focus on BO-optimized protocols and best performers
    warm_start_protocols = [
        # Aggressive BO (best Q30 from baselines)
        {'C1': 3.0, 'C2': 0.0, 'C3': 3.0},
        
        # Conservative BO (best LLI from baselines)
        {'C1': 0.91, 'C2': 0.0, 'C3': 0.54},
        
        # Middle ground
        {'C1': 2.0, 'C2': 0.0, 'C3': 2.0},
        {'C1': 1.5, 'C2': 1.5, 'C3': 1.5},
    ]
    
    print("\nAdding warm-start trials...")
    for params in warm_start_protocols:
        # Evaluate
        raw_data = evaluate_protocol(params)

        # Extract values BEFORE complete_trial (which converts to tuples)
        q30_val = raw_data['charge_throughput']
        lli_val = raw_data['total_lli']
        sei_val = raw_data['sei_growth']

        # Attach to experiment
        _, trial_index = ax_client.attach_trial(parameters=params)  # Returns (params, index)
        ax_client.complete_trial(trial_index=trial_index, raw_data=raw_data)

        # Print using saved values (complete_trial converts raw_data to tuples internally)
        print(f"  Trial {trial_index}: {params} -> Q30={q30_val:.6f} Ah, "
              f"LLI={lli_val:.9f} mol, SEI={sei_val:.2f} nm")
    
    print(f"\n✓ Added {len(warm_start_protocols)} warm-start trials")
    
except FileNotFoundError:
    print("⚠ No baseline results found. Starting from scratch.")
    print("  Recommendation: Run notebook 1 (baseline_protocols) first.")

In [ ]:
# Optimization settings
# num_iterations = 50
# trials_per_iteration = 3
num_iterations = 50
trials_per_iteration = 3
total_trials = num_iterations * trials_per_iteration

print("="*80)
print("MULTI-OBJECTIVE BAYESIAN OPTIMIZATION")
print("="*80)
print(f"Iterations: {num_iterations}")
print(f"Trials per iteration: {trials_per_iteration}")
#print(f"Total evaluations: {total_trials} (+ {ax_client.generation_strategy.num_running_trials_this_step} warm-start)")
print(f"Expected time: ~{total_trials * 0.5:.0f} - {total_trials * 1:.0f} minutes")
print("="*80 + "\n")

# Run optimization
for i in tqdm(range(num_iterations), desc="Optimization Progress"):
    # Generate new trial parameters
    trials, _ = ax_client.get_next_trials(max_trials=trials_per_iteration)  # Returns (trials_dict, is_complete)
    
    # Evaluate each trial
    for trial_index, parameters in trials.items():
        # Run simulation and get objectives
        print(parameters)
        raw_data = evaluate_protocol(parameters)
        
        # Complete trial
        ax_client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    
    # Print progress every 10 iterations
    if (i + 1) % 10 == 0:
        print(f"\n[Iteration {i+1}/{num_iterations}] Completed {(i+1)*trials_per_iteration} trials")
        
        # Get current Pareto frontier
        try:
            pareto_frontier = ax_client.get_pareto_optimal_parameters()
            print(f"  Current Pareto frontier size: {len(pareto_frontier)}")
        except:
            print(f"  Pareto frontier: Not yet available")

print("\n" + "="*80)
print("✓ OPTIMIZATION COMPLETE")
print("="*80)


## 7. Extract and Analyze Results

In [ ]:
# Convert experiment to DataFrame
results_df = exp_to_df(ax_client.experiment)

print("\nOptimization Results Summary:")
print("="*80)
print(f"Total trials completed: {len(results_df)}")
print(f"Successful trials: {results_df['trial_status'].value_counts().get('COMPLETED', 0)}")
print(f"Failed trials: {results_df['trial_status'].value_counts().get('FAILED', 0)}")

# Display objective ranges
print("\nObjective Ranges:")
print("-"*80)
for obj in ['charge_throughput', 'total_lli', 'sei_growth']:
    if obj in results_df.columns:
        print(f"{obj:20s}: {results_df[obj].min():.6f} - {results_df[obj].max():.6f}")

# Save full results
results_df.to_csv('results/multi_obj_all_trials.csv', index=False)
print("\n✓ Full results saved to results/multi_obj_all_trials.csv")

## 8. Extract Pareto Frontier

Identify non-dominated solutions:

In [ ]:
# Get Pareto-optimal parameters
try:
    pareto_frontier = ax_client.get_pareto_optimal_parameters()
    
    print("\n" + "="*80)
    print("PARETO FRONTIER")
    print("="*80)
    print(f"\nFound {len(pareto_frontier)} Pareto-optimal solutions:\n")
    
    # Extract Pareto points
    pareto_points = []
    
    # Structure: {trial_index: (parameters_dict, (objectives_dict, covariance_dict))}
    for trial_index, (parameters, (objectives, covariance)) in pareto_frontier.items():
        # Extract parameters
        c1 = parameters['C1']
        c2 = parameters['C2']
        c3 = parameters['C3']
        
        # Extract objectives
        q30 = objectives['charge_throughput']
        lli = objectives['total_lli']
        sei = objectives['sei_growth']
        
        # Store for export
        pareto_points.append({
            'trial_index': trial_index,
            'C1': c1,
            'C2': c2,
            'C3': c3,
            'charge_throughput': q30,
            'total_lli': lli,
            'sei_growth': sei
        })
    
    # Sort by Q30 for better readability
    pareto_points = sorted(pareto_points, key=lambda x: x['charge_throughput'], reverse=True)
    
    # Print top 5 and bottom 5
    print("Top 5 by Charge Throughput:")
    print("-"*80)
    for i, point in enumerate(pareto_points[:5], 1):
        print(f"\n{i}. Trial {point['trial_index']}:")
        print(f"   Protocol: [{point['C1']:.3f}, {point['C2']:.3f}, {point['C3']:.3f}]C")
        print(f"   Q30:       {point['charge_throughput']:.6f} Ah")
        print(f"   Total LLI: {point['total_lli']:.9f} mol")
        print(f"   SEI growth:{point['sei_growth']:.2f} nm")
    
    print("\n" + "-"*80)
    print("Bottom 5 by Charge Throughput (lowest degradation):")
    print("-"*80)
    for i, point in enumerate(pareto_points[-5:], 1):
        print(f"\n{i}. Trial {point['trial_index']}:")
        print(f"   Protocol: [{point['C1']:.3f}, {point['C2']:.3f}, {point['C3']:.3f}]C")
        print(f"   Q30:       {point['charge_throughput']:.6f} Ah")
        print(f"   Total LLI: {point['total_lli']:.9f} mol")
        print(f"   SEI growth:{point['sei_growth']:.2f} nm")
    
    # Save Pareto frontier
    pareto_df = pd.DataFrame(pareto_points)
    pareto_df.to_csv('results/optimal_protocols/pareto_frontier.csv', index=False)
    
    with open('results/optimal_protocols/pareto_frontier.json', 'w') as f:
        json.dump(pareto_points, f, indent=2)
    
    print("\n" + "="*80)
    print(f"✓ Pareto frontier saved to results/optimal_protocols/")
    print(f"  {len(pareto_points)} optimal protocols saved")
    print("="*80)
    
except Exception as e:
    print(f"\n⚠ Could not extract Pareto frontier: {e}")
    print("This can happen if not enough trials completed successfully.")
    import traceback
    traceback.print_exc()

## 9. Compare to Baselines

How do Pareto-optimal solutions compare to baseline protocols?

In [ ]:
# Load baselines
try:
    baseline_df = pd.read_csv('results/baseline_results.csv')
    
    print("\n" + "="*80)
    print("COMPARISON TO BASELINES")
    print("="*80 + "\n")
    
    # Best from Pareto frontier for each objective
    if len(pareto_points) > 0:
        pareto_df_analysis = pd.DataFrame(pareto_points)
        
        print("Best from Pareto Frontier:")
        print("-"*80)
        
        # Best Q30
        best_q30_idx = pareto_df_analysis['charge_throughput'].idxmax()
        best_q30 = pareto_df_analysis.loc[best_q30_idx]
        print(f"  Highest Q30: {best_q30['charge_throughput']:.6f} Ah")
        print(f"    Protocol: [{best_q30['C1']:.2f}, {best_q30['C2']:.2f}, {best_q30['C3']:.2f}]C")
        
        # Best LLI
        best_lli_idx = pareto_df_analysis['total_lli'].idxmin()
        best_lli = pareto_df_analysis.loc[best_lli_idx]
        print(f"\n  Lowest LLI: {best_lli['total_lli']:.9f} mol")
        print(f"    Protocol: [{best_lli['C1']:.2f}, {best_lli['C2']:.2f}, {best_lli['C3']:.2f}]C")
        
        # Best SEI
        best_sei_idx = pareto_df_analysis['sei_growth'].idxmin()
        best_sei = pareto_df_analysis.loc[best_sei_idx]
        print(f"\n  Lowest SEI: {best_sei['sei_growth']:.2f} nm")
        print(f"    Protocol: [{best_sei['C1']:.2f}, {best_sei['C2']:.2f}, {best_sei['C3']:.2f}]C")
        
        # Compare to baselines
        print("\n" + "-"*80)
        print("Improvement over Baselines:")
        print("-"*80)
        
        baseline_best_q30 = baseline_df['Q30'].max()
        baseline_best_lli = baseline_df['total_lli'].min()
        baseline_best_sei = baseline_df['sei_growth'].min()
        
        q30_improvement = (best_q30['charge_throughput'] - baseline_best_q30) / baseline_best_q30 * 100
        lli_improvement = (baseline_best_lli - best_lli['total_lli']) / baseline_best_lli * 100
        sei_improvement = (baseline_best_sei - best_sei['sei_growth']) / baseline_best_sei * 100
        
        print(f"  Q30:       {q30_improvement:+.2f}% vs best baseline")
        print(f"  Total LLI: {lli_improvement:+.2f}% vs best baseline")
        print(f"  SEI growth:{sei_improvement:+.2f}% vs best baseline")
        
        print("\n" + "="*80)
    
except FileNotFoundError:
    print("\n⚠ Baseline results not found. Run notebook 1 first for comparison.")

## 10. Save Experiment for Reproducibility

In [ ]:
# Save Ax experiment to JSON
ax_client.save_to_json_file('results/trials_database/ax_experiment.json')
print("✓ Ax experiment saved to results/trials_database/ax_experiment.json")

# Save configuration summary
config = {
    'experiment_name': 'fast_charge_multi_obj',
    'objectives': {
        'charge_throughput': 'maximize',
        'total_lli': 'minimize',
        'sei_growth': 'minimize'
    },
    'parameters': {
        'C1': [0.0, 3.0],
        'C2': [0.0, 3.0],
        'C3': [0.0, 3.0]
    },
    'protocol_structure': '3-step with 10-minute intervals',
    'soc_range': '10-90%',
    'degradation_modes': ['plating', 'SEI'],
    'num_iterations': num_iterations,
    'trials_per_iteration': trials_per_iteration,
    'total_trials': len(results_df),
    'pareto_frontier_size': len(pareto_points) if len(pareto_points) > 0 else 0
}

with open('results/trials_database/experiment_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Configuration saved to results/trials_database/experiment_config.json")
print("\n" + "="*80)
print("✓ ALL RESULTS SAVED")
print("="*80)
print("\nNext: Open notebook 4 (pareto_visualization) to create Pareto front plots")

---

## Summary

This notebook has:
1. ✓ Set up multi-objective Bayesian optimization with Ax Platform
2. ✓ Optimized 3-step fast-charging protocols for 3 competing objectives
3. ✓ Identified Pareto-optimal solutions balancing capacity and degradation
4. ✓ Compared results to baseline protocols
5. ✓ Saved all results for visualization and analysis

**Next Steps:**
- Notebook 4: Create interactive 3D Pareto front visualizations
- Analyze trade-offs between objectives
- Select protocols for experimental validation

---

### Issues/Limitations

- Solver does not converge for all parameter sets (ex: 3C ->3C->3C throws an error) despite these being valid conditions in practice

In [ ]:
evaluate_protocol({'C1': 3.0, 'C2': 1.6512603030122344, 'C3': 0.7735319125408446})